# Gemma 4 Developer Agent: a lean submission.zip builder, with the public evidence behind each setting

This notebook writes a complete, valid `submission.zip` for the **Gemma 4 Developer Agent** competition in a few seconds, with no GPU and no dataset download. It is a starting point you can fork and edit, and every non-default choice is tied to a public measurement so you can decide whether to keep it.

**Two agents from one flag.** `GRAPH_OPENING = False` builds **Agent A** (six workspace tools, grep-first locate). `GRAPH_OPENING = True` builds **Agent B**: the same agent plus `get_code_neighbors` and `search_similar_code`, with an instruction that opens with the shortlist-first sequence from the paper-track study linked below (identifiers copied verbatim from the report as seeds, 3-5 graph calls, then read only the top files). Everything else (budget, sampling, rules) is identical, so the two leaderboard scores are a paired test of the opening alone. Scores so far are in the last cell.

**What it builds**

* `agent.yaml`: one `LlmAgent`, six built-in tools, no sub-agents, no LoRA.
* `prompts/system.md`: a budget-aware system prompt (locate, read, reproduce, fix, verify, submit).
* `configs/sampling.yaml`: thinking on, thought summaries on.
* `eval_config.yaml`: per-task caps sized so 120 hidden tasks fit inside the 12-hour limit.

**Why these settings** (all numbers from public forum posts on the 129 public tasks with the 31B model; links at the end)

| Choice | Evidence |
|---|---|
| Keep thinking on, `include_thoughts: true` | Disabling thinking dropped resolved tasks from 48 to 23; hiding thought summaries dropped them to 35. |
| `thinking_budget: 4096` | 2,048 and 6,144 were not better. |
| `temperature: 1.0`, `top_p: 0.95` | 0.4 and 1.0 tied at 51/129, ahead of 0.2 (48) and 0.0 (47). |
| Keep ~50 tool calls | Halving to 25 calls removed 14 solved tasks. |
| `max_time_minutes` about 6 | Kaggle 4xL4 runs about 5.5 min/task; 120 tasks x 6 min is the ceiling that still fits in 12 h. |
| No graph tools, no analyzer sub-agent | Adding the three graph tools did not improve resolution (50 vs 55 without); every `search_similar_code` call returned empty; the analyzer sub-agent changed the count by one task. |

My own localization study explains the graph-tool result: the sandbox has no live encoder, so `search_similar_code` resolves the query against stored node ids and suffixes (with only a lexical fallback for free text); a model typing a paraphrase gets little or nothing back. Seeded with exact symbol names from the report, `get_code_neighbors` reaches 47.9% of true edit sites within two hops. Agent B is that recipe, written as instruction text.


In [ ]:
import os, zipfile, textwrap, pathlib

ROOT = pathlib.Path('/kaggle/working/submission')
for d in ['prompts', 'configs']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'   # or gemma-4-e4b-it-qat-w4a16-ct for the $10k small-model prize
GRAPH_OPENING = True   # False = Agent A (baseline), True = Agent B (graph tools + shortlist-first opening)
GRAPH_TOOLS = '- get_code_neighbors\n- search_similar_code\n' if GRAPH_OPENING else ''

(ROOT / 'agent.yaml').write_text(textwrap.dedent(f'''\
name: lean_swe_agent_{'b' if GRAPH_OPENING else 'a'}
model: {MODEL}
instruction: !include prompts/system.md
tools:
{GRAPH_TOOLS}- run_command
- read_file
- edit_file
- write_file
- get_status
- submit_patch
generate_content_config: !include configs/sampling.yaml
'''))

(ROOT / 'configs' / 'sampling.yaml').write_text(textwrap.dedent('''\
temperature: 1.0
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
'''))

(ROOT / 'eval_config.yaml').write_text(textwrap.dedent('''\
evaluation:
  timeout_seconds: 120
  max_tool_calls: 50
  max_time_minutes: 6
  max_turns: 100
'''))
print(open(ROOT / 'agent.yaml').read())


In [ ]:
SYSTEM_A = '''\
You are an expert Python maintainer. You are inside a git checkout of the repository at /workspace, with its dependencies already installed. Resolve the issue below with a minimal, correct source change. Hidden unit tests will be run against your patch, so the behaviour must match what the issue asks for exactly (parameter names, defaults, return values, exception types, error messages).

## Issue
{problem_description}

## Maintainer notes
{hints}

## Budget
You have about 6 minutes and 50 tool calls. Spend them like this: 1-4 calls to locate, 2-4 to read, 1-3 to edit, 2-5 to test, then submit_patch. Do not wander. Do not re-read files you already read.

## Method
1. Locate. Pull identifiers out of the issue (function, class, parameter, module, error text). Find them with one command, for example:
   run_command: grep -rn "def name_here\\|class Name" --include=*.py . | grep -v test | head -20
   If the issue is only a reference like "Fixes #1234", infer the subject from the maintainer notes and the repository, then grep the most specific words you have.
2. Read. Open only the relevant function and the 30-60 lines around it with read_file. Check whether the same logic exists in a sibling (sync/async pair, a second class, a second code path), because the fix usually belongs in all of them.
3. Reproduce (when cheap). Write a tiny script to /tmp/repro.py with write_file and run it with python. Never create files under /workspace.
4. Fix. Make the smallest change that satisfies the issue with edit_file. Keep each edit under 40 lines. Follow the surrounding style; do not refactor, rename, reformat or add comments elsewhere. Keep public behaviour that the issue does not mention unchanged.
5. Verify. Re-run /tmp/repro.py, then run the closest existing test file:
   python -m pytest tests/<file>.py -x -q 2>&1 | tail -15
   If something that passed before now fails, your change is wrong: fix it, do not hack the test.
6. Submit. Run: git status --short, confirm only intended source files changed (no test files, no pytest.ini, no conftest.py, no stray files), then call submit_patch. It is free and must be your last action. Always submit something: an imperfect fix scores better than nothing.

## Rules
- No internet, no pip install.
- Never edit tests, pytest.ini or conftest.py.
- Keep command output short: pipe to head or tail.
- If a tool call fails, change the approach instead of repeating it.
- When you implement a new option or parameter, add it to every signature in the chain it passes through, with the exact name from the issue.
'''

SYSTEM_B = '''\
You are an expert Python maintainer. You are inside a git checkout of the repository at /workspace, with its dependencies already installed. Resolve the issue below with a minimal, correct source change. Hidden unit tests will be run against your patch, so the behaviour must match what the issue asks for exactly (parameter names, defaults, return values, exception types, error messages).

## Issue
{problem_description}

## Maintainer notes
{hints}

## Budget
You have about 6 minutes and 50 tool calls. Spend them like this: 4-6 calls to locate (graph tools plus one grep), 2-4 to read, 1-3 to edit, 2-5 to test, then submit_patch. Do not wander. Do not re-read files you already read.

## Method
1. Locate with the code graph first (shortlist-first opening, 4-6 calls). This repository has a pre-built call graph. Bug reports rarely name the exact function that must change, but they almost always name a neighbour of it, so:
   a. Pull every identifier out of the issue and maintainer notes: function names, class names, parameter names, module names, names in tracebacks and error text. Keep them exactly as written.
   b. Call get_code_neighbors on the 2-3 most specific identifiers, passing the identifier as written (e.g. "Class.method" or "func_name"; the tool resolves short names). The callers and callees it returns are your candidate list; the fix is usually within one or two hops of a mentioned symbol.
   c. Call search_similar_code once with the single most specific identifier (a symbol name, not a sentence). Nodes returned by BOTH tools go to the top of the list.
   d. Take the top candidates, map each dotted node id to its file (fastapi.routing.func -> fastapi/routing.py), and open only the best 1-3 files. Sibling code paths (sync/async, subclasses) are usually also edited by the real fix.
   e. Add ONE keyword search so you are never relying on the graph alone (graph and keywords find different edit sites, and the best shortlist is their union):
   run_command: grep -rn "rare_word_1\\|rare_word_2\\|def name_here\\|class Name" --include=*.py . | grep -v test | head -20
   Use the rarest words from the issue (identifiers, error text, option names), not common ones. Rank files like this: files that appear in both the graph list and the grep output first, then the graph-only files, then the grep-only files. If the tools return nothing useful, do two more targeted greps instead.
   If the issue is only a reference like "Fixes #1234", infer the subject from the maintainer notes and the repository, then use the most specific words you have.
2. Read. Open only the relevant function and the 30-60 lines around it with read_file. Check whether the same logic exists in a sibling (sync/async pair, a second class, a second code path), because the fix usually belongs in all of them.
3. Reproduce (when cheap). Write a tiny script to /tmp/repro.py with write_file and run it with python. Never create files under /workspace.
4. Fix. Make the smallest change that satisfies the issue with edit_file. Keep each edit under 40 lines. Follow the surrounding style; do not refactor, rename, reformat or add comments elsewhere. Keep public behaviour that the issue does not mention unchanged.
5. Verify. Re-run /tmp/repro.py, then run the closest existing test file:
   python -m pytest tests/<file>.py -x -q 2>&1 | tail -15
   If something that passed before now fails, your change is wrong: fix it, do not hack the test.
6. Submit. Run: git status --short, confirm only intended source files changed (no test files, no pytest.ini, no conftest.py, no stray files), then call submit_patch. It is free and must be your last action. Always submit something: an imperfect fix scores better than nothing.

## Rules
- No internet, no pip install.
- Never edit tests, pytest.ini or conftest.py.
- Keep command output short: pipe to head or tail.
- If a tool call fails, change the approach instead of repeating it.
- When you implement a new option or parameter, add it to every signature in the chain it passes through, with the exact name from the issue.
'''

SYSTEM = SYSTEM_B if GRAPH_OPENING else SYSTEM_A
(ROOT / 'prompts' / 'system.md').write_text(SYSTEM)
print('Agent', 'B' if GRAPH_OPENING else 'A', '-', len(SYSTEM.split()), 'words')


In [ ]:
# Sanity checks the harness also performs: allowed extensions, agent.yaml at root, model in the registry
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
MODELS = {'gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-12b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct'}
import yaml
files = [p for p in ROOT.rglob('*') if p.is_file()]
assert all(p.suffix in ALLOWED for p in files), [p for p in files if p.suffix not in ALLOWED]
cfg = yaml.safe_load((ROOT / 'agent.yaml').read_text().replace('!include', ''))
assert cfg['model'] in MODELS, cfg['model']
for p in files:
    if p.suffix in {'.yaml', '.yml'}:
        yaml.safe_load(p.read_text().replace('!include', ''))

out = pathlib.Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.relative_to(ROOT))
print(f'{out} ({out.stat().st_size} bytes)')
for p in sorted(files): print(' ', p.relative_to(ROOT))


## How to submit

Download `submission.zip` from the Output tab (or use the Kaggle CLI), then **Submit Prediction > File Upload** on the competition page. You get one submission per day, so change one thing at a time and write it in the description.

## Leaderboard results of these exact files (public half of the hidden set)

| Agent | Model | Budget | Public score | Submitted |
|---|---|---|---|---|
| A | 31B | 6 min / 50 calls | 0.05 | Oct 25 |
| A | E4B | 5 min / 45 calls | 0.01 | Oct 26 |
| B | 31B | 6 min / 50 calls | pending | Oct 28 |
| B | E4B | 5 min / 45 calls | pending | Oct 29 |

Treat differences of a few tasks as noise: an independent study found 15% of outcomes flip between two identical runs (discussion 747168).

## Ideas to fork

1. **Small-model prize.** Set `MODEL` to `gemma-4-e4b-it-qat-w4a16-ct`; one of your two final selections must use E2B/E4B to be eligible for the extra $10,000.
2. **Deterministic localization skill.** Move Agent B's seed step out of the prompt into `skills/locate/` with a script that greps identifiers from the issue and maps them to dotted node ids before any model call.
3. **Time budget.** If your own calibration shows a faster median, raise `max_tool_calls` before raising `max_time_minutes`; the call budget was the stronger lever in the ablations.

## Sources

* Thakur, A. *Gemma 4 agent experiments: sampling, thinking, time limits, and tool ablations* (competition discussion 746250).
* Zhu, R. *Stronger models barely beat Gemma 4 31B in this harness* (discussion 747168).
* HARNESS_README.md in the competition data (section 10, "Competitor Best Practices and Important Gotchas").
* My paper-track study on localization signals: [Code-Graph Bug Localization Benchmark](https://www.kaggle.com/code/alikalwar/code-graph-bug-localization-benchmark-gemma-4-pap).

If this saved you a day of setup, an upvote helps others find it. Questions and results from your own runs are welcome in the comments.
